In [1]:
!pip install -q MDAnalysis MDTraj torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Define the file paths based on the uploaded files
WT_pdb = '/content/drive/MyDrive/PATH PASTE HERE /mdnvt_WT-rep1_firstframe.pdb'  # Replace with your PDB file name
WT_dcd = '/content/drive/MyDrive/ PATH PASTE HERE /mdnvt_WT-rep1.dcd'

In [4]:
import os
output_dir = '/content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/COLAB_UPLOADED'
os.makedirs(output_dir, exist_ok=True)

print(f"Working directory created at: {output_dir}")

Working directory created at: /content/drive/MyDrive/ABE_RF_Diffusion/GNN_AC/HIVPR/METHODS/COLAB_UPLOADED


In [5]:
import os
import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis.dihedrals import Ramachandran
from tqdm import tqdm

import torch
from torch_geometric.data import Data
from scipy.spatial.distance import pdist, squareform

In [6]:
PROT_RANGES = list(range(1, 199))

u = mda.Universe(WT_pdb, WT_dcd)

# Select Cα atoms in the target residue range
ca_atoms = u.select_atoms(
    "protein and name CA and resid " + " ".join(map(str, PROT_RANGES))
)
ca_resids = ca_atoms.resids           # node order is defined by these residues
n_residues = len(ca_resids)

print(f"Number of CA atoms (nodes): {n_residues}")
print("First 10 CA resids:", ca_resids[:10])

Number of CA atoms (nodes): 198
First 10 CA resids: [ 1  2  3  4  5  6  7  8  9 10]


/usr/local/lib/python3.12/dist-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


In [7]:
# ============================================================
# Part 2: Compute RMSF per Cα (over trajectory)
# ============================================================

# Collect coordinates over trajectory for selected CA atoms
coords_all = []

for ts in u.trajectory:
    coords_all.append(ca_atoms.positions.copy())

coords_all = np.array(coords_all)   # shape: (n_frames, n_residues, 3)
mean_coords = coords_all.mean(axis=0)

# RMSF: sqrt of mean squared fluctuation from average position
rmsf = np.sqrt(((coords_all - mean_coords) ** 2).sum(axis=2).mean(axis=0))
# shape: (n_residues,)
print("RMSF for first 5 residues:", rmsf[:5])
print("RMSF shape:", rmsf.shape)

RMSF for first 5 residues: [1.7964818  0.92217654 0.7646732  0.7279313  0.57401425]
RMSF shape: (198,)


In [8]:
# ============================================================
# Part 3: Chain-aware Ramachandran computation
#          (handles chain breaks by residue numbering)
# ============================================================

# We will:
# 1. Collect protein residues in PROT_RANGES that have full backbone N, CA, C.
# 2. Sort them by resid.
# 3. Split into "chains" whenever resid jump != 1.
# 4. For each chain (length >= 3), build backbone and run Ramachandran.
# 5. Keep only interior residues (chain_resids[1:-1]) as those with both φ/ψ.

# Step 3.1: collect valid residues
valid_residues = []

for res in u.residues:
    if res.resid in PROT_RANGES:
        # require full backbone atoms
        bb = res.atoms.select_atoms("name N CA C")
        ca = res.atoms.select_atoms("protein and name CA")
        if bb.n_atoms == 3 and ca.n_atoms == 1:
            valid_residues.append(res)

# sort by resid
valid_residues = sorted(valid_residues, key=lambda r: r.resid)
valid_resids = [int(r.resid) for r in valid_residues]

print("Number of valid backbone residues:", len(valid_residues))
print("First 10 valid resids:", valid_resids[:10])
print("Last 10 valid resids:", valid_resids[-10:])

# Step 3.2: split into chains based on breaks in resid numbering
chains = []
current_chain = [valid_residues[0]]

for prev_res, res in zip(valid_residues[:-1], valid_residues[1:]):
    if res.resid == prev_res.resid + 1:
        # same chain (continuous residue numbering)
        current_chain.append(res)
    else:
        # chain break
        chains.append(current_chain)
        current_chain = [res]
chains.append(current_chain)  # add last chain

print(f"Detected {len(chains)} chains based on residue numbering.")
for i, chain in enumerate(chains):
    cr = [int(r.resid) for r in chain]
    print(f"  Chain {i+1}: length={len(chain)}, resids={cr[0]}..{cr[-1]}")

# Step 3.3: run Ramachandran per chain and collect interior residues
all_dihedral_resids = []
all_dihedral_angles = []

for chain_idx, chain_residues in enumerate(chains, start=1):
    if len(chain_residues) < 3:
        # too short to have interior residues with both φ/ψ
        continue

    # Build backbone AtomGroup for this chain: concatenated N, CA, C
    backbone_atoms = mda.core.groups.AtomGroup([], u)
    for res in chain_residues:
        bb = res.atoms.select_atoms("name N CA C")
        if bb.n_atoms == 3:
            backbone_atoms += bb

    if len(backbone_atoms) < 9:  # at least 3 residues * 3 atoms
        continue

    print(f"Running Ramachandran for chain {chain_idx} with {len(chain_residues)} residues...")

    rama = Ramachandran(backbone_atoms).run()
    angles = rama.angles  # shape: (n_frames, n_chain_dihed, 2)

    # interior residues in this chain
    chain_resids = [int(r.resid) for r in chain_residues]
    interior_resids = chain_resids[1:-1]  # drop first and last

    if angles.shape[1] != len(interior_resids):
        raise ValueError(
            f"Chain {chain_idx}: Ramachandran length mismatch: "
            f"{angles.shape[1]} vs {len(interior_resids)}"
        )

    all_dihedral_resids.extend(interior_resids)
    all_dihedral_angles.append(angles)

# Concatenate over chains
dihedrals = np.concatenate(all_dihedral_angles, axis=1)   # (n_frames, total_dihed_res, 2)
dihedral_resids = np.array(all_dihedral_resids, dtype=int)

print("Final dihedrals shape:", dihedrals.shape)         # (n_frames, 194, 2) for your case
print("Number of dihedral residues:", len(dihedral_resids))
print("First 10 dihedral_resids:", dihedral_resids[:10])
print("Last 10 dihedral_resids:", dihedral_resids[-10:])

# Step 3.4: build resid -> dihedral index mapping
residue_to_index = {int(resid): idx for idx, resid in enumerate(dihedral_resids)}
n_dihed = dihedrals.shape[1]

Number of valid backbone residues: 198
First 10 valid resids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last 10 valid resids: [189, 190, 191, 192, 193, 194, 195, 196, 197, 198]
Detected 1 chains based on residue numbering.
  Chain 1: length=198, resids=1..198
Running Ramachandran for chain 1 with 198 residues...


/usr/local/lib/python3.12/dist-packages/MDAnalysis/analysis/dihedrals.py:444: UserWarning: Cannot determine phi and psi angles for the first or last residues
  warnings.warn(


Final dihedrals shape: (4000, 196, 2)
Number of dihedral residues: 196
First 10 dihedral_resids: [ 2  3  4  5  6  7  8  9 10 11]
Last 10 dihedral_resids: [188 189 190 191 192 193 194 195 196 197]


/usr/local/lib/python3.12/dist-packages/MDAnalysis/analysis/dihedrals.py:574: DeprecationWarning: The `angle` attribute was deprecated in MDAnalysis 2.0.0 and will be removed in MDAnalysis 3.0.0. Please use `results.angles` instead
  warnings.warn(wmsg, DeprecationWarning)


In [9]:
# ============================================================
# Part 4: Build per-frame graphs with [coords, RMSF, φ, ψ] + LJ-like edges
# ============================================================

# Rewind trajectory for graph construction
u.trajectory.rewind()

data_list = []

for i, ts in enumerate(tqdm(u.trajectory, desc="Building graphs")):
    # --- Node features ---

    # positions: (n_residues, 3)
    coords = ca_atoms.positions.copy()

    # RMSF as (n_residues, 1) – same for all frames here
    rmsf_frame = rmsf.reshape(-1, 1)

    # φ/ψ per residue, respecting chain breaks and termini
    phi_psi_frame = []
    for resid in ca_resids:
        resid_int = int(resid)
        idx = residue_to_index.get(resid_int, None)
        if idx is None or idx < 0 or idx >= n_dihed:
          # OLD: phi_psi_frame.append([np.nan, np.nan])
           phi_psi_frame.append([0.0, 0.0])  # <-- use 0 instead of NaN
        else:
           phi_psi_frame.append(dihedrals[i, idx])

    #phi_psi_frame = []
    #for resid in ca_resids:
    #    resid_int = int(resid)
    #    idx = residue_to_index.get(resid_int, None)
    #    if idx is None or idx < 0 or idx >= n_dihed:
    #        # No valid φ/ψ (chain ends or break): use NaN or zeros
    #        # Choose one; here I'll use NaN so you can differentiate later.
    #        phi_psi_frame.append([np.nan, np.nan])
    #    else:
    #       phi_psi_frame.append(dihedrals[i, idx])

    phi_psi_frame = np.array(phi_psi_frame)  # (n_residues, 2)

    # Concatenate node features: [x, y, z, RMSF, φ, ψ]
    node_features = np.concatenate([coords, rmsf_frame, phi_psi_frame], axis=1)
    x = torch.tensor(node_features, dtype=torch.float)

    # --- Edge index + edge attributes ---
    dist_matrix = squareform(pdist(coords))  # (n_residues, n_residues)
    mask = (dist_matrix < 10.0) & (dist_matrix > 0.0)  # 10 Å cutoff, no self
    src, tgt = np.where(mask)
    edge_index = torch.tensor([src, tgt], dtype=torch.long)

    distances = dist_matrix[src, tgt] + 1e-6
    inv_d = 1.0 / distances
    inv_d6 = inv_d ** 6
    exp_d = np.exp(-distances)

    edge_attr_np = np.stack([inv_d, exp_d, inv_d6], axis=1)  # (E, 3)
    edge_attr = torch.tensor(edge_attr_np, dtype=torch.float)

    # --- Create graph object ---
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    data_list.append(data)

print(f"✅ Created {len(data_list)} graph frames")
print(f"Node feature shape (sample): {data_list[0].x.shape}")
print(f"Edge feature shape (sample): {data_list[0].edge_attr.shape}")

Building graphs:   0%|          | 0/4000 [00:00<?, ?it/s]/tmp/ipykernel_5608/1142149261.py:51: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src, tgt], dtype=torch.long)
Building graphs: 100%|██████████| 4000/4000 [00:06<00:00, 627.95it/s]

✅ Created 4000 graph frames
Node feature shape (sample): torch.Size([198, 6])
Edge feature shape (sample): torch.Size([3348, 3])


In [10]:
torch.save(data_list, os.path.join(output_dir, 'trajectory_graphs_phi_psi_rmsf_WT_rep1.pt'))